# Session 3 — scikit-learn: Leakage & Pipelines
**Phase 3+4 Combined | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Data leakage is the most dangerous bug in quantitative research. It makes garbage strategies look spectacular. It's responsible for a large fraction of published backtests that fail in live trading.

scikit-learn has built-in tools to prevent leakage — but only if you use them correctly. The Pipeline object is the key. The `fit` vs `transform` discipline is the law.

> ⚠️ **The Rule:** Anything computed on training data (means, standard deviations, model parameters) must NOT be computed using test data — not even partially.


---
## 1️⃣ The fit/transform Discipline

| Method | What it does | When to call it |
|--------|-------------|----------------|
| `.fit(X)` | Learns parameters FROM data (e.g., computes mean/std) | Training data ONLY |
| `.transform(X)` | Applies learned parameters TO data | Both train and test |
| `.fit_transform(X)` | Convenience: fit then transform in one call | Training data ONLY — NEVER on test |

**The leakage scenario:**
```python
# ❌ WRONG — StandardScaler sees test data during fit
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)   # Uses test set statistics!
X_train, X_test = split(X_scaled)        # Too late — already contaminated

# ✅ RIGHT — Scaler only learns from training data
X_train, X_test = split(X_all)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Learn on train
X_test_scaled  = scaler.transform(X_test)         # Apply to test — no fit!
```


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Build a simple return prediction dataset
# Features: lagged returns and rolling stats
spy = yf.download('SPY', start='2015-01-01', auto_adjust=True)['Close'].squeeze()
log_ret = np.log(spy / spy.shift(1))

# Feature engineering
features = pd.DataFrame(index=log_ret.index)
features['ret_1d']  = log_ret.shift(1)
features['ret_5d']  = log_ret.shift(1).rolling(5).mean()
features['ret_20d'] = log_ret.shift(1).rolling(20).mean()
features['vol_20d'] = log_ret.shift(1).rolling(20).std()
features['vol_60d'] = log_ret.shift(1).rolling(60).std()

# Target: next day return
target = log_ret

# Align and drop NaN
df = pd.concat([features, target.rename('target')], axis=1).dropna()
X = df.drop('target', axis=1).values
y = df['target'].values
dates = df.index

print(f"Dataset: {len(df)} observations, {X.shape[1]} features")
print(f"Date range: {dates[0].date()} to {dates[-1].date()}")


In [ ]:
# ═══════════════════════════════════════════════
# THE LEAKAGE DEMO — Run the same model twice
# ═══════════════════════════════════════════════

# Split: 80% train, 20% test
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# ─────────────────────────────────────────────
# VERSION 1: WRONG — leakage via scaling on full dataset
# ─────────────────────────────────────────────
scaler_leaky = StandardScaler()
X_all_scaled = scaler_leaky.fit_transform(X)   # ← LEAKAGE: test stats used in fit
X_train_leaky = X_all_scaled[:split_idx]
X_test_leaky  = X_all_scaled[split_idx:]

model_leaky = Ridge(alpha=1.0)
model_leaky.fit(X_train_leaky, y_train)
r2_leaky_train = r2_score(y_train, model_leaky.predict(X_train_leaky))
r2_leaky_test  = r2_score(y_test,  model_leaky.predict(X_test_leaky))

# ─────────────────────────────────────────────
# VERSION 2: CORRECT — no leakage
# ─────────────────────────────────────────────
scaler_clean = StandardScaler()
X_train_clean = scaler_clean.fit_transform(X_train)   # ← Learn on train only
X_test_clean  = scaler_clean.transform(X_test)          # ← Apply (don't fit) to test

model_clean = Ridge(alpha=1.0)
model_clean.fit(X_train_clean, y_train)
r2_clean_train = r2_score(y_train, model_clean.predict(X_train_clean))
r2_clean_test  = r2_score(y_test,  model_clean.predict(X_test_clean))

print("=== Leakage Demo Results ===")
print()
print(f"{'Version':<20} {'Train R²':>10} {'Test R²':>10}")
print("-" * 42)
print(f"{'Leaky (wrong)':<20} {r2_leaky_train:>10.6f} {r2_leaky_test:>10.6f}")
print(f"{'Clean (correct)':<20} {r2_clean_train:>10.6f} {r2_clean_test:>10.6f}")
print()
print("Key observation:")
print("  In this simple case the difference may look small.")
print("  With more features, rolling z-scores, or interaction terms,")
print("  leakage inflates test R² dramatically — making garbage look good.")
print()
print("  The bigger danger: rolling/expanding statistics computed on the")
print("  FULL time series before splitting — those carry future info backward.")


---
## 2️⃣ The Pipeline Pattern — Leakage-Proof by Design

`sklearn.Pipeline` chains transformers and a model together. When you call `pipeline.fit(X_train)`, it ONLY fits transformers on training data — the test set never touches the fit step.

This is the correct way to build any ML workflow. Not optional.

```
Pipeline([
    ('scaler', StandardScaler()),   ← fits ONLY on train data
    ('model', Ridge(alpha=1.0))     ← trains on scaled train data
])
```


In [ ]:
# Build a proper pipeline — leakage impossible by construction

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# Train on training data only
pipeline.fit(X_train, y_train)

# Evaluate
r2_pipe_train = r2_score(y_train, pipeline.predict(X_train))
r2_pipe_test  = r2_score(y_test,  pipeline.predict(X_test))

print("Pipeline results:")
print(f"  Train R²: {r2_pipe_train:.6f}")
print(f"  Test R²:  {r2_pipe_test:.6f}")
print()
print("The pipeline guarantees the scaler never sees test data.")
print("This matches the 'clean' manual version exactly.")
print(f"  Manual clean test R²:   {r2_clean_test:.6f}")
print(f"  Pipeline test R²:       {r2_pipe_test:.6f}")
print("  Match: ", abs(r2_clean_test - r2_pipe_test) < 1e-10)


---
## 3️⃣ Feature Engineering Leakage — The Subtle One

Scaling leakage is obvious once you know about it. The subtle leakage happens in feature engineering:

| Leaky Feature | Why it leaks | Fix |
|--------------|-------------|-----|
| z-score computed on full dataset | Future mean/std used for past observations | Compute rolling z-score using only past data |
| Normalise by dataset-wide max | Future max contaminates past features | Use expanding window max |
| Cross-sectional rank on full time series | Future rank structure used | Rank within each time period independently |


In [ ]:
# Subtle leakage: z-scoring return features using full-dataset stats
# This is what most people do and is technically wrong for backtesting

# ❌ WRONG: z-score using full dataset mean/std
ret_series = log_ret.dropna()
mean_all  = ret_series.mean()
std_all   = ret_series.std()
z_leaky   = (ret_series - mean_all) / std_all

# ✅ CORRECT: expanding window z-score (only uses past data at each point)
expanding_mean = ret_series.expanding(min_periods=60).mean()
expanding_std  = ret_series.expanding(min_periods=60).std()
z_clean        = (ret_series - expanding_mean) / expanding_std

print("Leaky z-score (full dataset):")
print(f"  Mean: {z_leaky.mean():.6f}  Std: {z_leaky.std():.6f}")
print()
print("Clean z-score (expanding window):")
print(f"  Mean: {z_clean.dropna().mean():.6f}  Std: {z_clean.dropna().std():.6f}")
print()
print("Difference in last 252 values (recent period):")
diff = (z_leaky - z_clean).abs().dropna().tail(252)
print(f"  Mean absolute difference: {diff.mean():.6f}")
print(f"  Max absolute difference:  {diff.max():.6f}")
print()
print("The difference grows as the full-dataset mean drifts from rolling mean.")
print("In trending markets this can be substantial.")


---
## ✅ Self-Check Questions

1. What's the difference between `.fit()` and `.transform()`?
   > *`.fit()` learns parameters from data (e.g., mean, std). `.transform()` applies those learned parameters. `.fit()` must only ever be called on training data.*

2. Why does scaling on the full dataset before splitting cause leakage?
   > *The scaler learns mean and std from test data too. When it scales the training data, it's using statistics that include future observations — the model implicitly "knows" the future distribution.*

3. What problem does `sklearn.Pipeline` solve?
   > *It guarantees transformers are only fit on training data. When you call `pipeline.fit(X_train)`, every step in the chain only sees training data. Leakage is structurally impossible.*

4. Give an example of feature engineering leakage that ISN'T scaling.
   > *Computing a rolling z-score using the full dataset mean/std. The historical mean implicitly includes future returns, biasing the feature.*

---
## 🧪 Optional Tasks

- Deliberately create a leaky pipeline (fit scaler on full dataset). Measure the performance gap vs clean pipeline. Does the gap widen with more features?
- Build a pipeline with: StandardScaler → PCA(n_components=3) → Ridge. Verify that PCA only sees training data.
- Implement the expanding-window z-score vs full-dataset z-score for a momentum signal. Compare the resulting signal series — are they materially different?
